In [3]:
from typing import List, TypedDict
from pydantic import BaseModel, Field, field_validator
import re
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

In [4]:
load_dotenv()

True

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)


/Users/manojpaudel/Desktop/C_RAG/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "rag_documents"

In [8]:
if os.path.exists(CHROMA_PATH) and os.listdir(CHROMA_PATH):
    print("[Chroma] Loading existing database from disk...")
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH,
    )
    print(f"[Chroma] Loaded {vector_store._collection.count()} chunks.")

else:
    print("[Chroma] Building database for the first time — this may take a minute...")

    docs = (
        PyPDFLoader("./documents/book1.pdf").load()
        + PyPDFLoader("./documents/book2.pdf").load()
        + PyPDFLoader("./documents/book3.pdf").load()
    )

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=900, chunk_overlap=150
    ).split_documents(docs)

    for d in chunks:
        d.page_content = (
            d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_PATH,
    )
    print(f"[Chroma] Saved {len(chunks)} chunks to ./chroma_db")

[Chroma] Loading existing database from disk...
[Chroma] Loaded 6684 chunks.


In [9]:
retriever = vector_store.as_retriever(
    search_type="similarity", search_kwargs={"k": 4}
)

In [28]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=512,
    api_key=os.getenv("GROQ_API_KEY"),
)

In [29]:
UPPER_TH = 0.7
LOWER_TH = 0.3

In [30]:
def is_garbage_chunk(text: str) -> bool:
    # Pattern 1: repetitive numeric sequences  e.g. "1.1.1.1.1.1"
    if re.search(r'(\d+\.){10,}', text):
        return True

    # Pattern 2: less than 30% real letters → mostly numbers/punctuation
    letters = len(re.findall(r'[a-zA-Z]', text))
    total   = len(text.strip())
    if total > 50 and letters / total < 0.30:
        return True

    # Pattern 3: too little meaningful text after stripping non-letters
    meaningful = re.sub(r'[^a-zA-Z\s]', '', text).strip()
    if len(meaningful) < 30:
        return True

    return False


def sanitize(text: str, max_chars: int = 800) -> str:
    # Remove angle-bracket patterns that confuse Groq's function-call parser
    text = re.sub(r"<[^>]{0,60}>", " ", text)
    # Collapse all whitespace to single spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Truncate to avoid token overflow in structured output calls
    return text[:max_chars]

In [31]:
class State(TypedDict):
    question:str
    docs:List[Document]
    good_docs:List[Document]
    verdict:str
    reason:str
    strips:List[str]
    kept_strips:List[str]
    refined_context:str
    web_query:str
    web_docs:List[Document]
    answer:str

In [32]:
def retrieve_node(state: State) -> State:
    q = state["question"]
    return {"docs": retriever.invoke(q)}


In [33]:
class DocEvalScore(BaseModel):
    score: float = Field(
        description="Relevance score between 0.0 (irrelevant) and 1.0 (perfectly answers the question)."
    )
    reason: str = Field(
        description="One-sentence explanation of the score."
    )

doc_eval_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict retrieval evaluator for RAG.\n"
     "You will be given ONE retrieved chunk and a question.\n"
     "Return a relevance score in [0.0, 1.0].\n"
     "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
     "- 0.0: chunk is irrelevant\n"
     "Be conservative with high scores.\n"
     "Also return a short reason."),
    ("human", "Question: {question}\n\nChunk:\n{chunk}"),
])

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)

In [34]:
def eval_each_doc_node(state: State) -> State:
    q      = state["question"]
    scores : List[float]    = []
    good   : List[Document] = []

    for d in state["docs"]:
        # Skip garbage chunks before they reach Groq
        if is_garbage_chunk(d.page_content):
            print("[eval_each_doc] skipping garbage chunk")
            scores.append(0.0)
            continue

        try:
            clean_chunk = sanitize(d.page_content, max_chars=800)
            out: DocEvalScore = doc_eval_chain.invoke({
                "question": q,
                "chunk":    clean_chunk,
            })
            scores.append(out.score)
            if out.score > LOWER_TH:
                good.append(d)

        except Exception as e:
            print(f"[eval_each_doc] skipping chunk due to error: {e}")
            scores.append(0.0)

    # CORRECT: at least one doc > UPPER_TH
    if any(s > UPPER_TH for s in scores):
        return {
            "good_docs": good,
            "verdict":   "CORRECT",
            "reason":    f"At least one retrieved chunk scored > {UPPER_TH}.",
        }

    # INCORRECT: all docs < LOWER_TH
    if len(scores) > 0 and all(s < LOWER_TH for s in scores):
        return {
            "good_docs": [],
            "verdict":   "INCORRECT",
            "reason":    f"All retrieved chunks scored < {LOWER_TH}.",
        }

    # AMBIGUOUS: otherwise
    return {
        "good_docs": good,
        "verdict":   "AMBIGUOUS",
        "reason":    f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}.",
    }


In [35]:
def decompose_to_sentences(text: str) -> List[str]:
    text      = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

In [36]:
class KeepOrDrop(BaseModel):
    keep: bool = Field(
        description="Must be a boolean true or false — NOT a string. "
                    "true if the sentence directly helps answer the question, false otherwise."
    )

    @field_validator("keep", mode="before")
    @classmethod
    def coerce_to_bool(cls, v):
        """Coerce string 'true'/'false' → real bool if Groq sends a string."""
        if isinstance(v, str):
            return v.strip().lower() == "true"
        return v

filter_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict relevance filter.\n"
     "Return keep=true only if the sentence directly helps answer the question.\n"
     "keep must be a boolean (true or false), NOT a string.\n"
     "Use ONLY the sentence."),
    ("human", "Question: {question}\n\nSentence:\n{sentence}"),
])

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

In [37]:
def refine(state: State) -> State:
    q = state["question"]

    if state.get("verdict") == "CORRECT":
        docs_to_use = state["good_docs"]
    elif state.get("verdict") == "INCORRECT":
        docs_to_use = state["web_docs"]
    else:  # AMBIGUOUS
        docs_to_use = state["good_docs"] + state["web_docs"]

    context = "\n\n".join(d.page_content for d in docs_to_use).strip()
    strips  = decompose_to_sentences(context)

    kept: List[str] = []
    for s in strips:
        # Skip garbage sentences before they reach Groq
        if is_garbage_chunk(s):
            continue

        try:
            clean_s = sanitize(s, max_chars=400)
            out: KeepOrDrop = filter_chain.invoke({
                "question": q,
                "sentence": clean_s,
            })
            if out.keep:
                kept.append(s)
        except Exception as e:
            print(f"[refine] skipping sentence due to error: {e}")
            continue

    refined_context = "\n".join(kept).strip()

    return {
        "strips":          strips,
        "kept_strips":     kept,
        "refined_context": refined_context,
    }

In [38]:
class WebQuery(BaseModel):
    query: str = Field(
        description="Short web search query (6-14 words) derived from the user question."
    )

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user question into a web search query composed of keywords.\n"
     "Rules:\n"
     "- Keep it short (6-14 words).\n"
     "- If the question implies recency (e.g., recent/latest/last week/last month), add a constraint like (last 30 days).\n"
     "- Do NOT answer the question.\n"
     "- Return JSON with a single key: query"),
    ("human", "Question: {question}"),
])

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery)

In [39]:
def rewrite_query_node(state: State) -> State:
    out: WebQuery = rewrite_chain.invoke({"question": state["question"]})
    return {"web_query": out.query}

In [40]:
tavily = TavilySearchResults(max_results=5)


def web_search_node(state: State) -> State:
    q       = state.get("web_query") or state["question"]
    results = tavily.invoke({"query": q})

    web_docs: List[Document] = []
    for r in results or []:
        title   = r.get("title", "")
        url     = r.get("url", "")
        content = r.get("content", "") or r.get("snippet", "")
        text    = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"
        web_docs.append(Document(page_content=text, metadata={"url": url, "title": title}))

    return {"web_docs": web_docs}

In [41]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
     "If the context is empty or insufficient, say: 'I don't know.'"),
    ("human", "Question: {question}\n\nContext:\n{context}"),
])


def generate(state: State) -> State:
    out = (answer_prompt | llm | StrOutputParser()).invoke({
        "question": state["question"],
        "context":  state["refined_context"],
    })
    return {"answer": out}

In [42]:
def route_after_eval(state: State) -> str:
    if state["verdict"] == "CORRECT":
        return "refine"
    else:
        return "rewrite_query"

In [43]:
g = StateGraph(State)

g.add_node("retrieve",      retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)

g.add_node("rewrite_query", rewrite_query_node)
g.add_node("web_search",    web_search_node)

g.add_node("refine",        refine)
g.add_node("generate",      generate)

g.add_edge(START,           "retrieve")
g.add_edge("retrieve",      "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {
        "refine":        "refine",
        "rewrite_query": "rewrite_query",
    },
)

# non-correct path
g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search",    "refine")

# correct path already goes to refine
g.add_edge("refine",        "generate")
g.add_edge("generate",      END)

app = g.compile()

In [ ]:
if __name__ == "__main__":
    res = app.invoke(
        {
            "question":"Batch normalization",
            "docs":[],
            "good_docs":[],
            "verdict":"",
            "reason":"",
            "strips":[],
            "kept_strips":[],
            "refined_context":"",
            "web_query":"",
            "web_docs":[],
            "answer":"",
        }
    )

    print("VERDICT:",   res["verdict"])
    print("REASON:",    res["reason"])
    print("WEB_QUERY:", res["web_query"])
    print("\nOUTPUT:\n", res["answer"])

VERDICT: CORRECT
REASON: At least one retrieved chunk scored > 0.7.
WEB_QUERY: 

OUTPUT:
 Batch normalization (BN) and layer normalization (LN) are both techniques used to normalize the inputs of a neural network, but they differ in their application and scope.

Batch normalization normalizes the inputs of each layer over a mini-batch, which means it standardizes the mean and variance of each unit within a mini-batch. This helps to stabilize the learning process and address the Internal Covariate Shift problem, where the distribution of each layer's inputs changes during training.

Layer normalization, on the other hand, normalizes the inputs of each layer over the entire sequence or sample, not just a mini-batch. This means it standardizes the mean and variance of each unit across the entire input sequence or sample.

In general, batch normalization is more commonly used in deep neural networks, where the Internal Covariate Shift problem is more pronounced. Layer normalization is ofte